In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

In [ ]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP_assignment1/'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

In [ ]:
API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())

# Do not hardcode login credentials in the notebook.
# In Colab, prefer Secrets named POLIMILLIONAIRE_USERNAME and POLIMILLIONAIRE_PASSWORD.
import getpass
import os

MANUAL_USERNAME = (__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip())
MANUAL_PASSWORD = (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip())
def read_notebook_secret(name: str) -> str:
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return ""


username = MANUAL_USERNAME or read_notebook_secret("POLIMILLIONAIRE_USERNAME") or read_notebook_secret("MILLIONAIRE_USERNAME")
password = MANUAL_PASSWORD or read_notebook_secret("POLIMILLIONAIRE_PASSWORD") or read_notebook_secret("MILLIONAIRE_PASSWORD")

if not username:
    username = input("PoliMillionaire username: ").strip()
if not password:
    password = getpass.getpass("PoliMillionaire password: ").strip()


In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")

In [ ]:
# Choose a competition ID
comp_id = 1

In [ ]:
def play_game(game):
  # Manual text-mode play helper for the updated API.
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      question_text = "" if getattr(question, "text", None) is None else str(question.text)
      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question_text}")
      print()

      for opt in question.options:
          opt_text = "" if getattr(opt, "text", None) is None else str(opt.text)
          print(f"  [{opt.id}] {opt_text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print("CORRECT!")
          if result.game_over:
              print("\nCONGRATULATIONS! You completed the game!")
              print(f"Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f"Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print("\nGame Over!")
        print(f"Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print("WRONG ANSWER!")
          print("\nGame Over!")
          print(f"Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")


In [ ]:
import json
import re
import html
from urllib.parse import quote, urlencode, urljoin, urlparse, parse_qs
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import time

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_USER_AGENT = "PoliMillionaireNLP/1.0 student project"
WIKIPEDIA_REQUEST_DELAY_SECONDS = 0.8
WIKIPEDIA_429_BACKOFF_SECONDS = 4.0
WIKIPEDIA_MAX_RETRIES = 2
MAX_WIKIPEDIA_SEARCH_QUERIES = 2
_LAST_WIKIPEDIA_REQUEST_TIME = 0.0
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "there", "these", "this", "those", "to", "was", "were",
    "what", "when", "where", "which", "who", "why", "with", "according", "article",
    "considered", "important", "goal", "goals", "main", "primary", "following"
}


def question_to_text(question) -> str:
    """Accept a string, a dict, or a millionaire_client Question object."""
    if hasattr(question, "text"):
        return "" if question.text is None else str(question.text)
    if isinstance(question, dict) and "text" in question:
        return "" if question["text"] is None else str(question["text"])
    return str(question)


def normalize_wikipedia_text(text: str) -> str:
    """Clean a plain Wikipedia extract enough for later NLP steps."""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", str(text).lower()) if len(token) > 1]


def expand_term(token: str) -> set[str]:
    """Tiny synonym/variant helper for common historical wording traps."""
    variants = {token}
    if token == "roman":
        variants.update({"rome", "romans"})
    elif token in {"rome", "romans"}:
        variants.add("roman")
    return variants


def extract_keywords(text: str, limit: int = 10) -> list[str]:
    keywords = []
    seen = set()
    for token in tokenize(text):
        if token in STOPWORDS or token in seen:
            continue
        keywords.append(token)
        seen.add(token)
        if len(keywords) >= limit:
            break
    return keywords


def capital_context_phrases(question_text: str) -> list[str]:
    """Build focused phrases like 'Roman marriage' from capitalized topic words."""
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", question_text)
    phrases = []
    for index, word in enumerate(words):
        if not word[:1].isupper() or word.lower() in STOPWORDS:
            continue
        phrase_words = [word]
        for next_word in words[index + 1:index + 4]:
            if next_word.lower() in STOPWORDS:
                break
            phrase_words.append(next_word)
        if len(phrase_words) > 1:
            phrases.append(" ".join(phrase_words))
    return phrases


def build_wikipedia_search_queries(question) -> list[str]:
    """Create several focused Wikipedia queries instead of trusting the full question only."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    keywords = extract_keywords(cleaned, limit=10)

    queries = []
    queries.extend(capital_context_phrases(cleaned))
    if keywords:
        queries.append(" ".join(keywords[:6]))
    if len(keywords) >= 2:
        queries.append(" ".join(keywords[:2]))
    queries.append(cleaned)
    queries.append(question_text)

    deduped = []
    seen = set()
    for query in queries:
        normalized = normalize_wikipedia_text(query).lower()
        if normalized and normalized not in seen:
            deduped.append(query)
            seen.add(normalized)
    return deduped


def wikipedia_request(params: dict, timeout: float = 6.0) -> dict:
    """Call the free MediaWiki API with delay and simple 429 backoff."""
    global _LAST_WIKIPEDIA_REQUEST_TIME

    url = f"{WIKIPEDIA_API}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": WIKIPEDIA_USER_AGENT})

    for attempt in range(WIKIPEDIA_MAX_RETRIES + 1):
        elapsed_since_last = time.monotonic() - _LAST_WIKIPEDIA_REQUEST_TIME
        sleep_for = WIKIPEDIA_REQUEST_DELAY_SECONDS - elapsed_since_last
        if sleep_for > 0:
            time.sleep(sleep_for)

        try:
            with urlopen(request, timeout=timeout) as response:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as exc:
            _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()
            if exc.code != 429 or attempt >= WIKIPEDIA_MAX_RETRIES:
                raise

            retry_after = exc.headers.get("Retry-After")
            try:
                wait_seconds = float(retry_after) if retry_after else WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)
            except ValueError:
                wait_seconds = WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)

            print(f"Wikipedia rate limit hit. Waiting {wait_seconds:.1f}s before retry {attempt + 1}/{WIKIPEDIA_MAX_RETRIES}...")
            time.sleep(wait_seconds)


def search_wikipedia(query: str, limit: int = 5, timeout: float = 6.0) -> list[dict]:
    """Search Wikipedia and return candidate pages for one query string."""
    query = normalize_wikipedia_text(query)
    data = wikipedia_request(
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": limit,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
    )

    results = data.get("query", {}).get("search", [])
    return [
        {
            "title": item.get("title", ""),
            "page_id": item.get("pageid"),
            "snippet": normalize_wikipedia_text(re.sub(r"<[^>]+>", " ", item.get("snippet", ""))),
            "query": query,
            "search_rank": rank,
        }
        for rank, item in enumerate(results, start=1)
    ]


def collect_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None) -> list[dict]:
    """Search capped focused queries and deduplicate candidate pages by title."""
    candidates_by_title = {}
    search_queries = build_wikipedia_search_queries(question)
    if max_search_queries is None:
        max_search_queries = MAX_WIKIPEDIA_SEARCH_QUERIES
    if max_search_queries is not None:
        search_queries = search_queries[:max_search_queries]
    for query in search_queries:
        try:
            results = search_wikipedia(query, limit=per_query_limit, timeout=timeout)
        except Exception as exc:
            print(f"Wikipedia search skipped for {query!r}: {exc}")
            continue

        for result in results:
            title_key = result["title"].lower()
            if title_key not in candidates_by_title:
                candidates_by_title[title_key] = result
            else:
                candidates_by_title[title_key]["search_rank"] = min(
                    candidates_by_title[title_key]["search_rank"],
                    result["search_rank"],
                )
    return list(candidates_by_title.values())


def candidate_relevance_score(candidate: dict, question) -> float:
    """Score title/snippet overlap with question keywords; penalize very generic one-word titles."""
    question_text = question_to_text(question)
    keywords = extract_keywords(question_text, limit=10)
    candidate_text = f"{candidate.get('title', '')} {candidate.get('snippet', '')}"
    candidate_terms = set(tokenize(candidate_text))

    matched = 0
    for keyword in keywords:
        if expand_term(keyword) & candidate_terms:
            matched += 1

    overlap = matched / max(1, len(keywords))
    title_terms = tokenize(candidate.get("title", ""))
    rank_bonus = 1.0 / max(1, candidate.get("search_rank", 1))
    generic_penalty = 0.35 if len(title_terms) == 1 and len(keywords) > 1 else 0.0

    return (1.6 * overlap) + (0.25 * rank_bonus) - generic_penalty


def rank_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None) -> list[dict]:
    candidates = collect_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries)
    for candidate in candidates:
        candidate["candidate_score"] = candidate_relevance_score(candidate, question)
    return sorted(candidates, key=lambda item: item["candidate_score"], reverse=True)


def fetch_wikipedia_extract(title: str, timeout: float = 6.0) -> dict:
    """Fetch a Wikipedia page as a plain-text document."""
    data = wikipedia_request(
        {
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "exsectionformat": "plain",
            "inprop": "url",
            "titles": title,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
    )

    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()), {}) if pages else {}
    return {
        "title": page.get("title", title),
        "page_id": page.get("pageid"),
        "url": page.get("fullurl") or f"https://en.wikipedia.org/wiki/{quote(title.replace(' ', '_'))}",
        "text": normalize_wikipedia_text(page.get("extract", "")),
        "source": "Wikipedia",
    }


def get_wikipedia_documents_for_question(question, top_n: int = 5, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None) -> list[dict]:
    """
    Return the top N related Wikipedia documents for a question.

    This avoids the trap of trusting only Wikipedia's first result for the full question.
    """
    query = question_to_text(question)
    candidates = rank_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries)
    documents = []

    for candidate in candidates[:top_n]:
        try:
            document = fetch_wikipedia_extract(candidate["title"], timeout=timeout)
        except Exception as exc:
            print(f"Wikipedia page skipped for {candidate['title']!r}: {exc}")
            continue

        document["query"] = query
        document["matched_query"] = candidate.get("query")
        document["search_rank"] = candidate.get("search_rank")
        document["candidate_score"] = candidate.get("candidate_score", 0.0)
        document["snippet"] = candidate.get("snippet", "")
        document["search_results"] = candidates
        documents.append(document)

    return documents


def get_wikipedia_documents_by_titles(titles: list[str], timeout: float = 6.0, query: str = "offline_index_titles") -> list[dict]:
    """Fetch exact Wikipedia pages for offline indexing without search-result ambiguity."""
    documents = []
    for rank, title in enumerate(titles, start=1):
        try:
            document = fetch_wikipedia_extract(title, timeout=timeout)
        except Exception as exc:
            print(f"Wikipedia exact page skipped for {title!r}: {exc}")
            continue

        if not document.get("text"):
            continue
        document["query"] = query
        document["matched_query"] = title
        document["search_rank"] = rank
        document["candidate_score"] = 1.0
        document["snippet"] = document.get("text", "")[:500]
        document["search_results"] = []
        documents.append(document)
    return documents


def get_wikipedia_document_for_question(question, search_limit: int = 5, timeout: float = 6.0) -> dict:
    """Backward-compatible helper: return only the highest-ranked document."""
    documents = get_wikipedia_documents_for_question(question, top_n=1, per_query_limit=search_limit, timeout=timeout)
    if documents:
        return documents[0]
    return {
        "query": question_to_text(question),
        "title": None,
        "page_id": None,
        "url": None,
        "text": "",
        "source": "Wikipedia",
        "search_results": [],
    }



# ---- Optional National Archives Catalog source for the RAG document pool ----
# This keeps retrieval focused on Wikipedia + Catalog only.

EXTRA_SOURCE_USER_AGENT = WIKIPEDIA_USER_AGENT
EXTRA_SOURCE_REQUEST_DELAY_SECONDS = 0.15
_LAST_EXTRA_SOURCE_REQUEST_TIME = {}
NARA_SEARCH_API = "https://catalog.archives.gov/api/v2/records/search"


def clean_source_text(text: str) -> str:
    """Clean snippets from JSON/API sources into compact plain text."""
    text = html.unescape(str(text or "")).replace("\xa0", " ")
    text = re.sub(r"(?is)<(script|style|noscript).*?</\1>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def iter_text_values(value, max_items: int = 20):
    """Flatten common API JSON values into text strings."""
    if value is None or max_items <= 0:
        return []
    if isinstance(value, str):
        text = clean_source_text(value)
        return [text] if text else []
    if isinstance(value, (int, float)):
        return [str(value)]
    if isinstance(value, dict):
        values = []
        for item in value.values():
            values.extend(iter_text_values(item, max_items=max_items - len(values)))
            if len(values) >= max_items:
                break
        return values
    if isinstance(value, (list, tuple, set)):
        values = []
        for item in value:
            values.extend(iter_text_values(item, max_items=max_items - len(values)))
            if len(values) >= max_items:
                break
        return values
    text = clean_source_text(value)
    return [text] if text else []


def join_text_fields(*values, max_chars: int = 3500) -> str:
    parts = []
    seen = set()
    for value in values:
        for text in iter_text_values(value):
            key = text.lower()
            if text and key not in seen:
                parts.append(text)
                seen.add(key)
            if sum(len(part) for part in parts) >= max_chars:
                break
    return clean_source_text(". ".join(parts))[:max_chars]


def get_secret_value(name: str, default: str = "") -> str:
    """Read a secret from environment variables or Colab Secrets."""
    import os

    value = os.environ.get(name)
    if value:
        return value

    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    return default


def resolve_national_archives_api_key(api_key: str = "") -> str:
    """Resolve the hardcoded Catalog API key."""
    return (api_key or globals().get("NATIONAL_ARCHIVES_API_KEY", "")).strip()


def national_archives_catalog_headers(api_key: str) -> dict:
    """Headers required by the National Archives Catalog API support email."""
    return {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }


def source_json_request(url: str, timeout: float = 4.0, source_name: str = "source", headers=None, delay=None) -> dict:
    global _LAST_EXTRA_SOURCE_REQUEST_TIME

    if delay is None:
        delay = EXTRA_SOURCE_REQUEST_DELAY_SECONDS
    last_time = _LAST_EXTRA_SOURCE_REQUEST_TIME.get(source_name, 0.0)
    sleep_for = delay - (time.monotonic() - last_time)
    if sleep_for > 0:
        time.sleep(sleep_for)

    request_headers = {
        "User-Agent": EXTRA_SOURCE_USER_AGENT,
        "Accept": "application/json,text/plain,*/*",
    }
    if headers:
        request_headers.update(headers)

    request = Request(url, headers=request_headers)
    with urlopen(request, timeout=timeout) as response:
        _LAST_EXTRA_SOURCE_REQUEST_TIME[source_name] = time.monotonic()
        raw_text = response.read().decode("utf-8", "replace")
        content_type = response.headers.get("Content-Type", "").lower()
        if "json" not in content_type and not raw_text.lstrip().startswith(("{", "[")):
            raise ValueError(f"{source_name} did not return JSON; content-type={content_type!r}")
        return json.loads(raw_text)


def build_catalog_search_query(question, max_terms: int = 7) -> str:
    """Build one compact query for the National Archives Catalog."""
    question_text = question_to_text(question)
    phrases = capital_context_phrases(question_text)
    keywords = extract_keywords(question_text, limit=max_terms)
    if phrases:
        phrase_terms = set(tokenize(phrases[0]))
        tail = [keyword for keyword in keywords if keyword not in phrase_terms]
        query = " ".join([phrases[0], *tail[:max_terms]])
    elif keywords:
        query = " ".join(keywords[:max_terms])
    else:
        query = question_text
    return clean_source_text(query)


def build_catalog_search_queries(question, max_terms: int = 7) -> list[str]:
    """Create focused Catalog query variants, similar to the Wikipedia search step."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = clean_source_text(cleaned)
    keywords = extract_keywords(cleaned, limit=max_terms)

    queries = []
    queries.extend(capital_context_phrases(cleaned))
    compact = build_catalog_search_query(question, max_terms=max_terms)
    if compact:
        queries.append(compact)
    if keywords:
        queries.append(" ".join(keywords[:min(4, len(keywords))]))
    queries.append(cleaned)

    deduped = []
    seen = set()
    for query in queries:
        query = clean_source_text(query)
        key = query.lower()
        if query and key not in seen:
            deduped.append(query)
            seen.add(key)
    return deduped


def source_relevance_score(question, title: str, text: str, rank: int = 1) -> float:
    candidate = {"title": title or "", "snippet": (text or "")[:800], "search_rank": rank}
    return candidate_relevance_score(candidate, question)


def make_source_document(source: str, title: str, url: str, text: str, query: str, question, rank: int = 1, snippet: str = "") -> dict:
    text = clean_source_text(text)
    snippet = clean_source_text(snippet or text[:500])
    return {
        "source": source,
        "title": clean_source_text(title) or source,
        "url": url,
        "text": text,
        "query": question_to_text(question),
        "matched_query": query,
        "search_rank": rank,
        "candidate_score": source_relevance_score(question, title, f"{snippet} {text}", rank=rank),
        "snippet": snippet,
    }


def first_text(value, default: str = "") -> str:
    values = iter_text_values(value, max_items=1)
    return values[0] if values else default


def nara_candidate_records(data: dict) -> list:
    containers = [data]
    if isinstance(data, dict) and isinstance(data.get("body"), dict):
        containers.append(data["body"])
    for container in containers:
        for key in ("results", "records"):
            value = container.get(key) if isinstance(container, dict) else None
            if isinstance(value, list):
                return value
        hits = container.get("hits") if isinstance(container, dict) else None
        if isinstance(hits, list):
            return hits
        if isinstance(hits, dict):
            for key in ("hits", "records", "results"):
                value = hits.get(key)
                if isinstance(value, list):
                    return value
    return []


def nara_record_payload(hit):
    """Unwrap a National Archives Catalog v2 hit into the actual record payload."""
    if not isinstance(hit, dict):
        return {"text": hit}, {}

    source = hit.get("_source", hit)
    if not isinstance(source, dict):
        return {"text": source}, {}

    for key in ("record", "description", "item", "object"):
        payload = source.get(key)
        if isinstance(payload, dict):
            return payload, source

    return source, source


def catalog_title_values(value, max_items: int = 5) -> list[str]:
    """Extract human-readable titles/names from nested Catalog metadata without IDs or URLs."""
    results = []

    def visit(item):
        if len(results) >= max_items or item is None:
            return
        if isinstance(item, str):
            text = clean_source_text(item)
            if text and not re.search(r"https?://|\.(?:jpg|jpeg|png|gif|pdf|tif|tiff)\b", text, flags=re.IGNORECASE):
                results.append(text)
            return
        if isinstance(item, dict):
            for key in ("title", "name", "heading", "displayName", "description"):
                text = clean_source_text(item.get(key))
                if text:
                    results.append(text)
                    return
            for key in ("parent", "ancestor", "record", "description"):
                if key in item:
                    visit(item.get(key))
                    return
            return
        if isinstance(item, (list, tuple)):
            for child in item:
                visit(child)
                if len(results) >= max_items:
                    break

    visit(value)
    deduped = []
    seen = set()
    for text in results:
        key = text.lower()
        if key not in seen:
            deduped.append(text)
            seen.add(key)
    return deduped[:max_items]


def catalog_scalar_text(value) -> str:
    """Return compact text only for simple Catalog values and string lists."""
    if value is None:
        return ""
    if isinstance(value, str):
        return clean_source_text(value)
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, (list, tuple)):
        parts = []
        for item in value:
            if isinstance(item, (str, int, float)):
                text = clean_source_text(item)
                if text:
                    parts.append(text)
            elif isinstance(item, dict):
                for key in ("termName", "name", "title", "displayName", "logicalDate", "year"):
                    text = clean_source_text(item.get(key))
                    if text:
                        parts.append(text)
                        break
        return clean_source_text("; ".join(dict.fromkeys(parts)))
    return ""


def build_catalog_rag_text(description: dict, title: str) -> str:
    """Build useful Catalog text for RAG while avoiding file names, URLs, IDs, and raw hierarchy metadata."""
    parts = []

    def add(label: str, value):
        text = catalog_scalar_text(value)
        if text:
            parts.append(f"{label}: {text}")

    if title:
        parts.append(f"Title: {title}")
    add("Scope note", description.get("scopeAndContentNote") or description.get("scopeContent"))
    add("Production dates", description.get("productionDateArray") or description.get("productionDates"))
    start_date = catalog_scalar_text(description.get("inclusiveStartDate"))
    end_date = catalog_scalar_text(description.get("inclusiveEndDate"))
    if start_date or end_date:
        parts.append(f"Inclusive dates: {start_date} to {end_date}".strip())
    add("Level", description.get("levelOfDescription"))
    add("Record type", description.get("recordType"))
    add("Materials", description.get("typeOfMaterials") or description.get("generalRecordsTypeArray") or description.get("generalRecordsTypes"))

    parent_titles = catalog_title_values(description.get("ancestors"), max_items=4)
    if parent_titles:
        parts.append("Collection context: " + "; ".join(parent_titles))

    text = clean_source_text(". ".join(parts))
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\b\S+\.(?:jpg|jpeg|png|gif|pdf|tif|tiff)\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip(" .")
    return text


def catalog_document_from_hit(hit, query: str, question, rank: int = 1):
    description, source_record = nara_record_payload(hit)
    metadata = source_record.get("metadata", {}) if isinstance(source_record, dict) else {}
    control_group = metadata.get("controlGroup", {}) if isinstance(metadata, dict) else {}
    title = first_text(
        description.get("title") or description.get("itemTitle") or description.get("localIdentifier"),
        "National Archives Catalog result",
    )
    naid = first_text(description.get("naId") or description.get("naid") or description.get("identifierNaid") or control_group.get("naId"))
    url = f"https://catalog.archives.gov/id/{naid}" if naid else "https://catalog.archives.gov/"
    text = build_catalog_rag_text(description, title)
    if not text and title:
        text = title
    if not text:
        return None
    return make_source_document("National Archives Catalog", title, url, text, query, question, rank=rank)


def get_national_archives_catalog_documents_for_question(
    question,
    api_key: str = "",
    top_n: int = 1,
    timeout: float = 4.0,
    per_query_limit=None,
    max_search_queries=1,
) -> list[dict]:
    """Fetch limited National Archives Catalog documents. Requires a Catalog API key."""
    api_key = (api_key or "").strip()
    if not api_key:
        print("National Archives Catalog skipped: paste your key into NATIONAL_ARCHIVES_API_KEY.")
        return []

    per_query_limit = max(1, int(per_query_limit or top_n or 1))
    final_doc_limit = max(1, int(top_n or 1))
    queries = build_catalog_search_queries(question)
    if max_search_queries is not None:
        queries = queries[:max(1, int(max_search_queries))]

    headers = national_archives_catalog_headers(api_key)
    documents = []
    for query_index, query in enumerate(queries, start=1):
        data = source_json_request(
            f"{NARA_SEARCH_API}?{urlencode({'q': query, 'limit': per_query_limit})}",
            timeout=timeout,
            source_name="National Archives Catalog",
            headers=headers,
        )
        for rank, hit in enumerate(nara_candidate_records(data)[:per_query_limit], start=1):
            doc = catalog_document_from_hit(hit, query, question, rank=rank + ((query_index - 1) * per_query_limit))
            if doc:
                documents.append(doc)

    documents = dedupe_documents(documents)
    documents.sort(key=lambda doc: doc.get("candidate_score", 0), reverse=True)
    return documents[:final_doc_limit]


def test_national_archives_catalog_api(api_key: str = "", query: str = "Roman Empire", timeout: float = 5.0) -> dict:
    """Run one quick Catalog request to confirm the hardcoded key works."""
    api_key = resolve_national_archives_api_key(api_key)
    if not api_key:
        raise RuntimeError("Paste your Catalog key into NATIONAL_ARCHIVES_API_KEY first.")
    data = source_json_request(
        f"{NARA_SEARCH_API}?{urlencode({'q': query, 'limit': 1})}",
        timeout=timeout,
        source_name="National Archives Catalog",
        headers=national_archives_catalog_headers(api_key),
    )
    records = nara_candidate_records(data)
    print(f"National Archives Catalog API OK. Records returned: {len(records)}")
    return data


# Backward-compatible name used by earlier cells.
def get_national_archives_documents_for_question(question, api_key: str = "", top_n: int = 1, timeout: float = 4.0) -> list[dict]:
    return get_national_archives_catalog_documents_for_question(question, api_key=api_key, top_n=top_n, timeout=timeout)


def dedupe_documents(documents: list[dict]) -> list[dict]:
    unique = []
    seen = set()
    for doc in documents:
        key = (doc.get("source", ""), doc.get("url") or doc.get("title") or "")
        if key in seen:
            continue
        seen.add(key)
        unique.append(doc)
    return unique


def get_multi_source_documents_for_question(
    question,
    top_n: int = 1,
    per_query_limit: int = 2,
    timeout: float = 3.0,
    max_search_queries=None,
    use_wikipedia: bool = True,
    extra_source_top_n: int = 1,
    max_total_docs: int = 2,
    extra_source_timeout: float = 3.0,
    max_extra_source_seconds: float = 4.0,
    source_order=None,
    use_catalog: bool = True,
    use_national_archives=None,
    national_archives_api_key: str = "",
    catalog_max_search_queries=1,
    catalog_per_query_limit=None,
    **ignored_sources,
) -> list[dict]:
    """Retrieve only Wikipedia documents plus optional National Archives Catalog docs."""
    documents = []
    if use_wikipedia and top_n > 0:
        documents.extend(
            get_wikipedia_documents_for_question(
                question,
                top_n=top_n,
                per_query_limit=per_query_limit,
                timeout=timeout,
                max_search_queries=max_search_queries,
            )
        )

    catalog_enabled = use_catalog if use_national_archives is None else bool(use_national_archives)
    if catalog_enabled and extra_source_top_n > 0:
        extra_start = time.monotonic()
        if max_extra_source_seconds is None or time.monotonic() - extra_start < max_extra_source_seconds:
            try:
                documents.extend(
                    get_national_archives_catalog_documents_for_question(
                        question,
                        api_key=national_archives_api_key,
                        top_n=extra_source_top_n,
                        timeout=extra_source_timeout,
                        per_query_limit=catalog_per_query_limit,
                        max_search_queries=catalog_max_search_queries,
                    )
                )
            except Exception as exc:
                print(f"National Archives Catalog skipped: {exc}")

    documents = dedupe_documents(documents)
    if max_total_docs:
        documents = documents[:max_total_docs]
    return documents


# Example after starting a game:
# game = client.game.start(competition_id=comp_id)  # add mode="text" if your client supports it
# doc = get_wikipedia_document_for_question(game.current_question)
# print(doc["title"])
# print(doc["url"])
# print(doc["text"][:1000])


In [ ]:
import math


def split_sentences(text: str) -> list[str]:
    """Sentence splitter for clean Wikipedia text."""
    text = normalize_wikipedia_text(text)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [sentence.strip() for sentence in sentences if len(sentence.strip()) >= 40]


def build_rag_chunks(documents: list[dict], sentences_per_chunk: int = 5, overlap: int = 2) -> list[dict]:
    """Split retrieved source documents into overlapping evidence chunks."""
    chunks = []
    step = max(1, sentences_per_chunk - overlap)

    for doc_index, doc in enumerate(documents):
        doc_text = normalize_wikipedia_text(doc.get("text", ""))
        sentences = split_sentences(doc_text)
        if not sentences and len(doc_text) >= 30:
            sentences = [doc_text]
        for start in range(0, len(sentences), step):
            chunk_sentences = sentences[start:start + sentences_per_chunk]
            if not chunk_sentences:
                break
            chunk_text = " ".join(chunk_sentences)
            if len(chunk_text) < 40:
                continue
            chunks.append(
                {
                    "doc_index": doc_index,
                    "chunk_index": len(chunks),
                    "title": doc.get("title", ""),
                    "source": doc.get("source", "Wikipedia"),
                    "url": doc.get("url", ""),
                    "text": chunk_text,
                    "document_score": float(doc.get("candidate_score", 0.0)),
                }
            )
            if start + sentences_per_chunk >= len(sentences):
                break

    return chunks


def lexical_similarity(query: str, text: str) -> float:
    """Fallback score if sklearn is unavailable."""
    query_terms = set(extract_keywords(query, limit=20))
    text_terms = set(tokenize(text))
    if not query_terms or not text_terms:
        return 0.0
    overlap = len(query_terms & text_terms) / len(query_terms)
    return overlap


def retrieve_rag_chunks(question, documents: list[dict], top_k: int = 8) -> list[dict]:
    """Retrieve the strongest chunks from the top-N documents for the question."""
    question_text = question_to_text(question)
    chunks = build_rag_chunks(documents)
    if not chunks:
        return []

    chunk_texts = [f"{chunk.get('source', '')} {chunk['title']} {chunk['text']}" for chunk in chunks]

    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        matrix = vectorizer.fit_transform([question_text] + chunk_texts)
        similarities = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
    except Exception:
        similarities = [lexical_similarity(question_text, text) for text in chunk_texts]

    ranked = []
    for chunk, similarity in zip(chunks, similarities):
        score = float(similarity) + 0.08 * chunk.get("document_score", 0.0)
        enriched = dict(chunk)
        enriched["retrieval_score"] = score
        ranked.append(enriched)

    ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return ranked[:top_k]


def build_offline_rag_index(
    documents: list[dict],
    sentences_per_chunk: int = 5,
    chunk_overlap: int = 2,
    index_name: str = "test3_offline_rag_index",
) -> dict:
    """Pre-index a document store into chunks and a local TF-IDF vector matrix."""
    chunks = build_rag_chunks(documents, sentences_per_chunk=sentences_per_chunk, overlap=chunk_overlap)
    chunk_texts = [f"{chunk.get('source', '')} {chunk.get('title', '')} {chunk.get('text', '')}" for chunk in chunks]
    index = {
        "name": index_name,
        "documents": documents,
        "chunks": chunks,
        "chunk_texts": chunk_texts,
        "sentences_per_chunk": sentences_per_chunk,
        "chunk_overlap": chunk_overlap,
        "retriever": "lexical",
        "vectorizer": None,
        "matrix": None,
        "created_at": None,
    }

    try:
        from datetime import datetime, timezone
        index["created_at"] = datetime.now(timezone.utc).isoformat()
    except Exception:
        pass

    if chunk_texts:
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer

            vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
            matrix = vectorizer.fit_transform(chunk_texts)
            index["vectorizer"] = vectorizer
            index["matrix"] = matrix
            index["retriever"] = "tfidf_vector_index"
        except Exception as exc:
            index["retriever"] = f"lexical_fallback:{exc}"

    return index


def save_offline_rag_index(index: dict, path) -> None:
    """Persist the prebuilt index so later runs can load it before the timed game."""
    import pickle
    from pathlib import Path

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as handle:
        pickle.dump(index, handle)


def load_offline_rag_index(path) -> dict:
    """Load a persisted offline index."""
    import pickle

    with open(path, "rb") as handle:
        return pickle.load(handle)


def offline_rag_index_summary(index: dict) -> dict:
    """Small serializable summary for logs and debug prints."""
    if not index:
        return {"available": False}
    return {
        "available": True,
        "name": index.get("name"),
        "documents": len(index.get("documents", [])),
        "chunks": len(index.get("chunks", [])),
        "retriever": index.get("retriever"),
        "created_at": index.get("created_at"),
        "sentences_per_chunk": index.get("sentences_per_chunk"),
        "chunk_overlap": index.get("chunk_overlap"),
    }


def retrieve_offline_rag_chunks(question, index: dict, top_k: int = 8) -> list[dict]:
    """Retrieve top chunks from a prebuilt offline index without calling external APIs."""
    if not index or not index.get("chunks"):
        return []

    question_text = question_to_text(question)
    chunks = index.get("chunks", [])
    chunk_texts = index.get("chunk_texts") or [
        f"{chunk.get('source', '')} {chunk.get('title', '')} {chunk.get('text', '')}"
        for chunk in chunks
    ]

    vectorizer = index.get("vectorizer")
    matrix = index.get("matrix")
    if vectorizer is not None and matrix is not None:
        try:
            from sklearn.metrics.pairwise import cosine_similarity

            query_vector = vectorizer.transform([question_text])
            similarities = cosine_similarity(query_vector, matrix).flatten()
        except Exception:
            similarities = [lexical_similarity(question_text, text) for text in chunk_texts]
    else:
        similarities = [lexical_similarity(question_text, text) for text in chunk_texts]

    ranked = []
    for chunk, similarity in zip(chunks, similarities):
        score = float(similarity) + 0.08 * chunk.get("document_score", 0.0)
        enriched = dict(chunk)
        enriched["retrieval_score"] = score
        enriched["retrieval_mode"] = "offline_index"
        ranked.append(enriched)

    ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return ranked[:top_k]


def documents_from_offline_hits(index: dict, hits: list[dict], max_docs: int = 5) -> list[dict]:
    """Return the source documents represented by retrieved offline chunks."""
    documents = index.get("documents", []) if index else []
    selected = []
    seen = set()
    for hit in hits:
        doc_index = hit.get("doc_index")
        if doc_index is None or doc_index in seen:
            continue
        if 0 <= doc_index < len(documents):
            selected.append(documents[doc_index])
            seen.add(doc_index)
        if len(selected) >= max_docs:
            break
    return selected


def build_rag_context(hits: list[dict], max_chars: int = 4200) -> str:
    """Format retrieved chunks as compact evidence for generation."""
    blocks = []
    used = 0
    for index, hit in enumerate(hits, start=1):
        block = f"[Evidence {index} | {hit.get('source', 'Source')}: {hit['title']}] {hit['text']}"
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def rank_answer_sentences(question, hits: list[dict], max_sentences: int = 5) -> list[str]:
    """Extract the most relevant evidence sentences for a no-LLM answer."""
    question_text = question_to_text(question)
    question_terms = set(tokenize(question_text))
    purpose_terms = {"goal", "purpose", "reason", "important", "considered", "used", "use", "tool", "primarily", "primary", "fundamental", "institution"}
    wants_purpose = bool(question_terms & purpose_terms)
    candidates = []
    seen = set()

    for hit_index, hit in enumerate(hits):
        for sentence_index, sentence in enumerate(split_sentences(hit.get("text", ""))):
            key = sentence.lower()
            if key in seen:
                continue
            seen.add(key)
            sentence_terms = set(tokenize(sentence))
            sentence_for_score = f"{hit.get('source', '')} {hit.get('title', '')} {sentence}"
            score = lexical_similarity(question_text, sentence_for_score) + 0.15 * hit.get("retrieval_score", 0.0)
            if wants_purpose:
                score += 0.25 * len(sentence_terms & purpose_terms)
            if hit.get("title", "").lower() in sentence.lower():
                score += 0.05
            candidates.append((score, hit_index, sentence_index, sentence))

    candidates.sort(key=lambda item: item[0], reverse=True)
    return [sentence for _, _, _, sentence in candidates[:max_sentences]]


def extractive_rag_answer(question, hits: list[dict], max_sentences: int = 5) -> str:
    """Create a concise explanation paragraph from retrieved evidence sentences."""
    sentences = rank_answer_sentences(question, hits, max_sentences=max_sentences)
    if not sentences:
        return "I could not find enough evidence in the retrieved Wikipedia documents to answer confidently."
    return " ".join(sentences)


DEEPSEEK_MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
_DEEPSEEK_CACHE = {}


def get_huggingface_token():
    """Read a Hugging Face token from Colab Secrets or environment variables, without hard-coding it."""
    import os

    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        token = os.environ.get(name)
        if token:
            return token

    try:
        from google.colab import userdata
        for name in ("hf_token", "HF_TOKEN", "huggingface"):
            token = userdata.get(name)
            if token:
                return token
    except Exception:
        pass

    return None


def load_deepseek_transformers_model(model_name: str = DEEPSEEK_MODEL_ID):
    """Load DeepSeek-compatible model locally. Uses 4-bit quantization on CUDA when available."""
    if model_name in _DEEPSEEK_CACHE:
        return _DEEPSEEK_CACHE[model_name]

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    token = get_huggingface_token()
    tokenizer_kwargs = {"trust_remote_code": True}
    model_kwargs = {"trust_remote_code": True}
    if token:
        tokenizer_kwargs["token"] = token
        model_kwargs["token"] = token

    tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            from transformers import BitsAndBytesConfig
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            model_kwargs.update({"quantization_config": quant_config, "device_map": "auto", "low_cpu_mem_usage": True})
        except Exception:
            model_kwargs.update({"torch_dtype": torch.float16, "device_map": "auto", "low_cpu_mem_usage": True})
    else:
        model_kwargs.update({"torch_dtype": torch.float32, "low_cpu_mem_usage": True})

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if not torch.cuda.is_available():
        model.to("cpu")
    model.eval()

    _DEEPSEEK_CACHE[model_name] = (tokenizer, model)
    return tokenizer, model


def deepseek_rag_answer(question, hits: list[dict], model_name: str = DEEPSEEK_MODEL_ID, max_new_tokens: int = 100) -> str:
    """Generate an explanatory RAG answer with the local DeepSeek instruct model."""
    import torch

    tokenizer, model = load_deepseek_transformers_model(model_name)
    question_text = question_to_text(question)
    context = build_rag_context(hits)

    messages = [
        {
            "role": "system",
            "content": "You answer questions using only the provided evidence. Do not mention multiple-choice options. Write one concise explanatory paragraph. If the evidence is insufficient, say so.",
        },
        {
            "role": "user",
            "content": f"Question: {question_text}\n\nEvidence:\n{context}\n\nAnswer the question in one clear paragraph.",
        },
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def answer_question_from_rag_hits(
    question,
    hits: list[dict],
    use_local_generator: bool = True,
    generator_model: str = DEEPSEEK_MODEL_ID,
    generator_max_new_tokens: int = 100,
    extractive_max_sentences: int = 3,
    method_prefix: str = "rag",
) -> dict:
    """Generate the final RAG explanation from already-retrieved chunks."""
    method = f"{method_prefix}:extractive"

    if use_local_generator:
        try:
            answer = deepseek_rag_answer(question, hits, model_name=generator_model, max_new_tokens=generator_max_new_tokens)
            method = f"{method_prefix}:deepseek:{generator_model}"
        except Exception as exc:
            print(f"Local generator skipped, using extractive RAG instead: {exc}")
            answer = extractive_rag_answer(question, hits, max_sentences=extractive_max_sentences)
    else:
        answer = extractive_rag_answer(question, hits, max_sentences=extractive_max_sentences)

    return {
        "question": question_to_text(question),
        "answer": answer,
        "method": method,
        "evidence_chunks": hits,
    }


def answer_question_with_rag(
    question,
    documents: list[dict],
    top_k_chunks: int = 8,
    use_local_generator: bool = True,
    generator_model: str = DEEPSEEK_MODEL_ID,
    generator_max_new_tokens: int = 100,
    extractive_max_sentences: int = 3,
) -> dict:
    """
    Ask the question from RAG using the top-N documents, without using answer options.

    Returns an explanatory answer plus the retrieved evidence chunks.
    """
    hits = retrieve_rag_chunks(question, documents, top_k=top_k_chunks)
    return answer_question_from_rag_hits(
        question,
        hits,
        use_local_generator=use_local_generator,
        generator_model=generator_model,
        generator_max_new_tokens=generator_max_new_tokens,
        extractive_max_sentences=extractive_max_sentences,
        method_prefix="live_rag",
    )


In [ ]:
# Run this BEFORE starting a timed game.
# Hugging Face token setup:
# 1. In Colab, open the left sidebar key icon (Secrets).
# 2. Add a secret named HF_TOKEN with your Hugging Face token as the value.
# 3. Enable notebook access for that secret.
# Local fallback: this cell will ask for the token with getpass if no secret/env var is found.

import getpass
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

required_packages = [
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("huggingface_hub", "huggingface_hub"),
]
missing_packages = [package for package, module in required_packages if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

# Optional manual token path. Prefer Colab Secrets named HF_TOKEN.
# If you insist on hardcoding temporarily, put it in MANUAL_HF_TOKEN and run this cell once.
MANUAL_HF_TOKEN = ""

hf_token = get_huggingface_token() or MANUAL_HF_TOKEN.strip()
if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (input hidden): ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

if not get_huggingface_token():
    raise RuntimeError("No Hugging Face token found. Add HF_TOKEN in Colab Secrets, set MANUAL_HF_TOKEN, or enter it when prompted.")

DEEPSEEK_REPO_ID = "unsloth/DeepSeek-R1-Distill-Qwen-7B-GGUF"
DEEPSEEK_GGUF_FILENAME = "DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf"
DEEPSEEK_GGUF_LOCAL_PATH = "/content/gdrive/MyDrive/NLP_assignment/models/DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf"

def download_deepseek_gguf_if_needed():
    path = Path(DEEPSEEK_GGUF_LOCAL_PATH)
    if path.exists():
        return str(path)

    from huggingface_hub import hf_hub_download

    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {DEEPSEEK_GGUF_FILENAME} from {DEEPSEEK_REPO_ID}...")
    downloaded_path = hf_hub_download(
        repo_id=DEEPSEEK_REPO_ID,
        filename=DEEPSEEK_GGUF_FILENAME,
        local_dir=str(path.parent),
        local_dir_use_symlinks=False,
    )
    downloaded_path = Path(downloaded_path)
    if downloaded_path != path:
        try:
            if path.exists():
                path.unlink()
        except Exception:
            pass
        downloaded_path.replace(path)
    return str(path)


start_time = time.time()
deepseek_model_path = download_deepseek_gguf_if_needed()
print(f"Preloading DeepSeek-R1-Distill-Qwen-7B Q4_K_M from {deepseek_model_path} before the timed game...")
deepseek_tokenizer = None
deepseek_model = load_gguf_model(deepseek_model_path)

print(f"DeepSeek Q4_K_M model is loaded and ready in {time.time() - start_time:.1f}s.")
print("Now start the game / run the RAG answer cell.")


In [ ]:
import re

LETTERS = "ABCD"

# Core generation config: DeepSeek-R1-Distill-Qwen-7B Q4_K_M via GGUF, auto-downloaded in the preload cell.
GENERATION_BACKEND = globals().get("GENERATION_BACKEND", "llama_cpp")  # "llama_cpp" or "transformers"
GENERATION_MODEL_SOURCE = globals().get("GENERATION_MODEL_SOURCE", globals().get("DEEPSEEK_GGUF_LOCAL_PATH", "/content/gdrive/MyDrive/NLP_assignment/models/DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf"))
GENERATION_GGUF_PATH = globals().get("GENERATION_GGUF_PATH", GENERATION_MODEL_SOURCE)

_GGUF_MODEL_CACHE = {}


def option_text(option) -> str:
    value = option.text if hasattr(option, "text") else option.get("text", "")
    return "" if value is None else str(value)


def option_id(option) -> int:
    return option.id if hasattr(option, "id") else option["id"]


def parse_option_choice(text: str, option_count: int = 4):
    """Parse A-D or 0-3 from LLM output."""
    cleaned = str(text).strip().upper()

    letter_match = re.search(r"\b([A-D])\b", cleaned)
    if letter_match:
        index = LETTERS.index(letter_match.group(1))
        return index if index < option_count else None

    digit_match = re.search(r"\b([0-3])\b", cleaned)
    if digit_match:
        index = int(digit_match.group(1))
        return index if index < option_count else None

    return None


def resolve_model_name(model_name=None) -> str:
    if model_name:
        return str(model_name)
    return str(GENERATION_MODEL_SOURCE)


def is_gguf_model_source(model_source: str) -> bool:
    return str(model_source).lower().endswith(".gguf")


def load_gguf_model(model_path: str):
    """Load a local GGUF model through llama_cpp, with caching."""
    if model_path in _GGUF_MODEL_CACHE:
        return _GGUF_MODEL_CACHE[model_path]

    import importlib.util

    if importlib.util.find_spec("llama_cpp") is None:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"])
    from llama_cpp import Llama

    llm = Llama(
        model_path=model_path,
        n_ctx=4096,
        n_threads=8,
        n_gpu_layers=-1,
        verbose=False,
    )
    _GGUF_MODEL_CACHE[model_path] = llm
    return llm


def deepseek_generate_text(messages, model_name=None, max_new_tokens: int = 128, max_length: int = 2048) -> str:
    model_name = resolve_model_name(model_name)

    if GENERATION_BACKEND == "llama_cpp" or is_gguf_model_source(model_name) or is_gguf_model_source(GENERATION_GGUF_PATH):
        model_path = GENERATION_GGUF_PATH or model_name
        llm = load_gguf_model(model_path)
        prompt = ""
        for message in messages:
            role = message.get("role", "user")
            content = message.get("content", "")
            prompt += f"<{role}>\n{content}\n"
        prompt += "<assistant>\n"
        result = llm(
            prompt,
            max_tokens=max_new_tokens,
            temperature=0.0,
            top_p=1.0,
            stop=["</s>", "<|end|>", "<assistant>"]
        )
        return result["choices"][0]["text"].strip()

    tokenizer, model = load_deepseek_transformers_model(model_name)
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    import torch

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def format_options_for_prompt(options) -> str:
    return "\n".join(
        f"{LETTERS[index]}. {option_text(option)}"
        for index, option in enumerate(options)
    )


def build_chunk_evidence_for_choice(chunks: list[dict], max_chars: int = 3000) -> str:
    """Format top retrieved chunks for direct option selection."""
    blocks = []
    used = 0
    for index, chunk in enumerate(chunks, start=1):
        block = (
            f"[Chunk {index} | {chunk.get('source', 'Source')}: {chunk.get('title', 'Untitled')} | "
            f"score={float(chunk.get('retrieval_score', 0.0)):.3f}] {chunk.get('text', '')}"
        )
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def build_llm_answer_chunk(question, model_name=None) -> dict:
    """Ask a separate LLM for a direct answer without showing options."""
    model_name = resolve_model_name(model_name)
    question_text = question_to_text(question)
    messages = [
        {
            "role": "system",
            "content": "Answer the question from your own knowledge. Do not use or mention answer options. Write one short factual paragraph.",
        },
        {
            "role": "user",
            "content": f"Question:\n{question_text}\n\nGive a concise answer based only on your own knowledge.",
        },
    ]
    answer_text = deepseek_generate_text(messages, model_name=model_name, max_new_tokens=120, max_length=2048)
    return {
        "source": "llm knowledge",
        "title": "llm answer",
        "retrieval_score": 0.0,
        "text": answer_text,
    }


def choose_option_from_rag_chunks(question, options, chunks: list[dict], model_name=None) -> dict:
    """Ask one LLM call to select the answer directly from top evidence chunks."""
    model_name = resolve_model_name(model_name)
    question_text = question_to_text(question)
    options_text = format_options_for_prompt(options)
    evidence_text = build_chunk_evidence_for_choice(chunks)

    messages = [
        {
            "role": "system",
            "content": "Use only the provided evidence chunks. Choose the single best option. Reply with the option letter first (A/B/C/D), then one short reason.",
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question_text}\n\n"
                f"Options:\n{options_text}\n\n"
                f"Evidence chunks:\n{evidence_text}\n\n"
                "Return format: Letter - short reason."
            ),
        },
    ]

    model_output = deepseek_generate_text(messages, model_name=model_name, max_new_tokens=80, max_length=3072)
    selected_index = parse_option_choice(model_output, option_count=len(options))
    if selected_index is None:
        selected_index = 0
        model_output = f"fallback_parse_failure | raw={model_output}"

    selected_option = options[selected_index]
    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": model_output,
        "used_chunks": [
            {
                "source": chunk.get("source"),
                "title": chunk.get("title"),
                "retrieval_score": chunk.get("retrieval_score"),
                "text": chunk.get("text", "")[:700],
            }
            for chunk in chunks[:5]
        ],
    }

In [ ]:
# =========================
# ACTUAL GAME: Simple RAG chunks -> one LLM chooser
# =========================
# Flow per question:
# 1) Retrieve docs
# 2) Retrieve top chunks
# 3) Add a separate pure-KB 'llm answer' chunk
# 4) Give question + options + all chunks to one LLM
# 5) Submit option id

comp_id = 1  # Set your competition ID here.

RUN_ACTUAL_GAME = True
COMPETITION_ID = comp_id
GAME_MODE = "text"
MAX_QUESTIONS = None
SUBMIT_ANSWERS = True

# Retrieval settings
MAX_SEARCH_QUERIES = 2
PER_QUERY_LIMIT = 2
TOP_N_DOCS = 2
WIKIPEDIA_TIMEOUT = 3.0
WIKIPEDIA_DELAY_SECONDS = 0.3
WIKIPEDIA_BACKOFF_SECONDS = 0.5
WIKIPEDIA_RETRIES = 0
TOP_K_CHUNKS = 6

# Optional extra source (kept available but off by default)
USE_NATIONAL_ARCHIVES = False
EXTRA_SOURCE_ORDER = ["national_archives"]
CATALOG_MAX_SEARCH_QUERIES = 1
CATALOG_PER_QUERY_LIMIT = 2
EXTRA_SOURCE_TOP_N = 1
MAX_TOTAL_DOCS = 3
EXTRA_SOURCE_TIMEOUT = 4.0
MAX_EXTRA_SOURCE_SECONDS = 5.0
NATIONAL_ARCHIVES_API_KEY = ""

# LLM + timing
GENERATION_BACKEND = globals().get("GENERATION_BACKEND", "llama_cpp")
GENERATION_MODEL_SOURCE = globals().get("GENERATION_MODEL_SOURCE", globals().get("DEEPSEEK_MODEL_ID", "DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf"))
GENERATION_GGUF_PATH = globals().get("GENERATION_GGUF_PATH", "")
DEEPSEEK_MODEL_FOR_GAME = GENERATION_MODEL_SOURCE
MANUAL_HF_TOKEN = ""
PRELOAD_DEEPSEEK = True
QUESTION_TIME_BUFFER = 2.0
MIN_SECONDS_TO_ATTEMPT = 1.0

# Keep this as requested
DELAY_SUBMIT_FOR_WIKI_COOLDOWN = True
TARGET_SUBMIT_ELAPSED_SECONDS = 29
MIN_SECONDS_LEFT_AT_SUBMIT = 1
MAX_SUBMIT_WAIT_SECONDS = 20

SAVE_RUN_LOG = True
RUN_LOG_DIR = "/content/gdrive/MyDrive/NLP_assignment/test3_simple_rag_game_runs"
VERBOSE = True
DEBUG_API_TIMER = True

import getpass
import importlib.util
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def ensure_runtime_packages():
    required_packages = [
        ("transformers", "transformers"),
        ("accelerate", "accelerate"),
        ("bitsandbytes", "bitsandbytes"),
        ("scikit-learn", "sklearn"),
    ]
    if GENERATION_BACKEND == "llama_cpp" or str(GENERATION_GGUF_PATH).lower().endswith(".gguf"):
        required_packages.append(("llama-cpp-python", "llama_cpp"))
    missing = [package for package, module in required_packages if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


def ensure_hf_token_for_game():
    if GENERATION_BACKEND == "llama_cpp" or str(GENERATION_GGUF_PATH).lower().endswith(".gguf"):
        return

    token = get_huggingface_token() or MANUAL_HF_TOKEN.strip()
    if not token:
        token = getpass.getpass("Hugging Face token (input hidden): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    if not get_huggingface_token():
        raise RuntimeError("No Hugging Face token found. Add HF_TOKEN in Colab Secrets, set MANUAL_HF_TOKEN, or enter it when prompted.")


def preload_deepseek_for_game():
    ensure_runtime_packages()
    ensure_hf_token_for_game()
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    print(f"Preloading {DEEPSEEK_MODEL_FOR_GAME} before starting timed game... (backend={GENERATION_BACKEND})")
    start = time.time()

    if GENERATION_BACKEND == "llama_cpp" or str(GENERATION_GGUF_PATH).lower().endswith(".gguf"):
        _ = load_gguf_model(GENERATION_GGUF_PATH or GENERATION_MODEL_SOURCE)
        print(f"GGUF model ready in {time.time() - start:.1f}s. Starting game only after this point.")
        return

    tokenizer, model = load_deepseek_transformers_model(DEEPSEEK_MODEL_FOR_GAME)

    import torch
    warmup_messages = [
        {"role": "system", "content": "Answer briefly."},
        {"role": "user", "content": "Say ready."},
    ]
    warmup_prompt = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(f"DeepSeek ready in {time.time() - start:.1f}s. Starting game only after this point.")


def seconds_available(game) -> float:
    remaining = game.time_remaining
    if remaining is None:
        return 30.0
    return max(0.0, float(remaining))


def fallback_option(question):
    return question.options[0]


def wait_before_submit_for_cooldown(game) -> float:
    if not DELAY_SUBMIT_FOR_WIKI_COOLDOWN:
        return 0.0

    current_remaining = seconds_available(game)
    target_remaining = max(MIN_SECONDS_LEFT_AT_SUBMIT, 30.0 - TARGET_SUBMIT_ELAPSED_SECONDS)
    wait_seconds = current_remaining - target_remaining
    wait_seconds = min(MAX_SUBMIT_WAIT_SECONDS, max(0.0, wait_seconds))

    if wait_seconds > 0:
        print(f"Answer ready. Waiting {wait_seconds:.1f}s before submit to give APIs cooldown time.")
        time.sleep(wait_seconds)

    return wait_seconds


def start_text_game_session(competition_id: int):
    if GAME_MODE != "text":
        raise RuntimeError("This runner is adapted for text mode.")

    try:
        return client.game.start(competition_id=competition_id, mode=GAME_MODE)
    except TypeError as exc:
        message = str(exc)
        if "mode" not in message and "unexpected keyword" not in message:
            raise
        print("Client wrapper does not support mode= yet; trying raw text-mode start request.")
        try:
            from millionaire_client.game import GameSession
            from millionaire_client.models import GameState

            response = client.game._client.post(
                "/api/game/start",
                data={"competitionId": competition_id, "mode": GAME_MODE},
            )
            return GameSession(client.game._client, GameState.from_dict(response))
        except Exception as raw_exc:
            print(f"Raw text-mode start failed ({raw_exc}); falling back to old start().")
            return client.game.start(competition_id=competition_id)


def api_timer_debug_snapshot(game):
    deadline = getattr(game.state, "question_deadline", None)
    now = datetime.now(deadline.tzinfo) if deadline is not None else datetime.now(timezone.utc)
    raw_seconds = (deadline - now).total_seconds() if deadline is not None else None
    snapshot = {
        "requested_game_mode": GAME_MODE,
        "returned_game_mode": getattr(game, "mode", None),
        "question_deadline": deadline.isoformat() if deadline is not None else None,
        "local_now": now.isoformat(),
        "deadline_minus_now_seconds": raw_seconds,
        "game_time_remaining": game.time_remaining,
    }

    print("\nAPI timer debug")
    print("requested GAME_MODE:", snapshot["requested_game_mode"])
    print("returned game.mode:", snapshot["returned_game_mode"])
    print("questionDeadline:", snapshot["question_deadline"])
    print("local now:", snapshot["local_now"])
    print("deadline - now seconds:", None if raw_seconds is None else round(raw_seconds, 2))
    print("game.time_remaining:", None if game.time_remaining is None else round(game.time_remaining, 2))
    return snapshot


def answer_one_question_simple(question, game=None) -> dict:
    start = time.monotonic()
    seconds_left_start = seconds_available(game) if game is not None else None

    documents = get_multi_source_documents_for_question(
        question,
        top_n=TOP_N_DOCS,
        per_query_limit=PER_QUERY_LIMIT,
        timeout=WIKIPEDIA_TIMEOUT,
        max_search_queries=MAX_SEARCH_QUERIES,
        extra_source_top_n=EXTRA_SOURCE_TOP_N,
        max_total_docs=MAX_TOTAL_DOCS,
        extra_source_timeout=EXTRA_SOURCE_TIMEOUT,
        max_extra_source_seconds=MAX_EXTRA_SOURCE_SECONDS,
        source_order=EXTRA_SOURCE_ORDER,
        use_catalog=USE_NATIONAL_ARCHIVES,
        national_archives_api_key=NATIONAL_ARCHIVES_API_KEY,
        catalog_max_search_queries=CATALOG_MAX_SEARCH_QUERIES,
        catalog_per_query_limit=CATALOG_PER_QUERY_LIMIT,
    )
    after_docs = time.monotonic()

    chunks = retrieve_rag_chunks(question, documents, top_k=TOP_K_CHUNKS)
    after_chunks = time.monotonic()

    llm_answer_chunk = build_llm_answer_chunk(question, model_name=DEEPSEEK_MODEL_FOR_GAME)
    chunks_with_llm = chunks + [llm_answer_chunk]
    after_llm_answer = time.monotonic()

    option_match = choose_option_from_rag_chunks(
        question,
        question.options,
        chunks_with_llm,
        model_name=DEEPSEEK_MODEL_FOR_GAME,
    )
    after_choice = time.monotonic()

    return {
        "question": question_to_text(question),
        "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
        "documents": [
            {
                "source": doc.get("source"),
                "title": doc.get("title"),
                "url": doc.get("url"),
                "candidate_score": doc.get("candidate_score"),
                "matched_query": doc.get("matched_query"),
                "preview": doc.get("text", "")[:500],
            }
            for doc in documents
        ],
        "top_chunks": [
            {
                "source": chunk.get("source"),
                "title": chunk.get("title"),
                "retrieval_score": chunk.get("retrieval_score"),
                "text": chunk.get("text", "")[:900],
            }
            for chunk in chunks
        ],
        "llm_answer_chunk": {
            "source": llm_answer_chunk.get("source"),
            "title": llm_answer_chunk.get("title"),
            "retrieval_score": llm_answer_chunk.get("retrieval_score"),
            "text": llm_answer_chunk.get("text", "")[:900],
        },
        "chunks_sent_to_chooser": [
            {
                "source": chunk.get("source"),
                "title": chunk.get("title"),
                "retrieval_score": chunk.get("retrieval_score"),
                "text": chunk.get("text", "")[:900],
            }
            for chunk in chunks_with_llm
        ],
        "option_match": option_match,
        "elapsed_seconds": after_choice - start,
        "timings": {
            "document_retrieval_seconds": after_docs - start,
            "chunk_retrieval_seconds": after_chunks - after_docs,
            "llm_answer_seconds": after_llm_answer - after_chunks,
            "llm_choice_seconds": after_choice - after_llm_answer,
            "total_seconds": after_choice - start,
        },
        "seconds_left_start": seconds_left_start,
        "seconds_left_end": seconds_available(game) if game is not None else None,
    }


def play_actual_simple_rag_game():
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES

    if PRELOAD_DEEPSEEK:
        preload_deepseek_for_game()

    if not RUN_ACTUAL_GAME:
        print("RUN_ACTUAL_GAME is False. Set it to True to start a real timed game.")
        return None, None

    game = start_text_game_session(COMPETITION_ID)
    timer_debug = api_timer_debug_snapshot(game) if DEBUG_API_TIMER else None
    run_log = {
        "session_id": game.session_id,
        "competition_id": COMPETITION_ID,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "api_timer_debug": timer_debug,
        "questions": [],
    }

    print(f"Started game session {game.session_id}. Competition {COMPETITION_ID}. Mode {getattr(game, 'mode', GAME_MODE)}.")
    question_count = 0
    correct_count = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            print("No active question returned by server.")
            break

        question_count += 1
        current_level = game.current_level
        time_left = seconds_available(game)

        print("\n" + "=" * 80)
        print(f"Question {question_count} | Level {current_level} | {time_left:.1f}s left")
        print(question_to_text(question))
        for index, opt in enumerate(question.options):
            print(f"  {LETTERS[index]}. [{option_id(opt)}] {option_text(opt)}")
        print("=" * 80)

        if time_left < MIN_SECONDS_TO_ATTEMPT:
            selected = fallback_option(question)
            prediction = {
                "question": question_to_text(question),
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "top_chunks": [],
                "llm_answer_chunk": None,
                "chunks_sent_to_chooser": [],
                "option_match": {
                    "answer_id": option_id(selected),
                    "answer_text": option_text(selected),
                    "answer_index": 0,
                    "letter": "A",
                    "model_output": "fallback_time_guard",
                },
                "elapsed_seconds": 0.0,
                "seconds_left_start": time_left,
                "seconds_left_end": time_left,
            }
        else:
            prediction = answer_one_question_simple(question, game=game)

        selected_id = prediction["option_match"]["answer_id"]
        selected_text = prediction["option_match"]["answer_text"]
        selected_letter = prediction["option_match"]["letter"]

        if VERBOSE:
            print("\nTop chunks used for answer:")
            for i, chunk in enumerate(prediction.get("top_chunks", [])[:5], start=1):
                source = chunk.get("source", "Source")
                title = chunk.get("title", "Untitled")
                score = chunk.get("retrieval_score", 0.0)
                print(f"  [{i}] {source} | {title} | score={score:.3f}")
            if prediction.get("llm_answer_chunk"):
                llm_chunk = prediction["llm_answer_chunk"]
                print("\nLLM answer chunk:")
                print(f"  {llm_chunk.get('title', 'llm answer')} | {llm_chunk.get('text', '')}")
            print("\nChosen option:", f"{selected_letter}. [{selected_id}] {selected_text}")
            print("Chooser output:", prediction["option_match"].get("model_output"))
            timings = prediction.get("timings", {})
            if timings:
                print(
                    "Timing:",
                    f"docs={timings.get('document_retrieval_seconds', 0):.2f}s",
                    f"chunks={timings.get('chunk_retrieval_seconds', 0):.2f}s",
                    f"llm_answer={timings.get('llm_answer_seconds', 0):.2f}s",
                    f"llm_choice={timings.get('llm_choice_seconds', 0):.2f}s",
                    f"total={timings.get('total_seconds', prediction['elapsed_seconds']):.2f}s",
                )

        result_payload = None
        if SUBMIT_ANSWERS:
            submission_wait_seconds = wait_before_submit_for_cooldown(game)
            prediction["submission_wait_seconds"] = submission_wait_seconds
            if submission_wait_seconds:
                print(f"Time left after cooldown wait: {seconds_available(game):.1f}s")

            if seconds_available(game) <= QUESTION_TIME_BUFFER:
                print("Warning: low time before submit; submitting selected option immediately.")

            result = game.answer(selected_id)
            result_payload = {
                "correct": result.correct,
                "timed_out": result.timed_out,
                "game_over": result.game_over,
                "earned_amount": result.earned_amount,
            }
            correct_count += int(bool(result.correct))

            if result.correct:
                print(f"Correct. Earned: {result.earned_amount}")
            elif result.timed_out:
                print(f"Timed out. Earned: {result.earned_amount}")
            else:
                print(f"Wrong. Earned: {result.earned_amount}")
        else:
            print("Dry run: answer not submitted.")

        run_log["questions"].append(
            {
                "number": question_count,
                "level": current_level,
                "prediction": prediction,
                "result": result_payload,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )

        if result_payload and result_payload.get("game_over"):
            break
        if MAX_QUESTIONS is not None and question_count >= MAX_QUESTIONS:
            print("MAX_QUESTIONS reached; stopping.")
            break
        if not SUBMIT_ANSWERS:
            break

    run_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    run_log["questions_answered"] = question_count
    run_log["correct_count"] = correct_count
    run_log["final_earned_amount"] = game.earned_amount

    if SAVE_RUN_LOG:
        log_dir = Path(RUN_LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"test3_simple_rag_game_{game.session_id}.json"
        with open(log_path, "w", encoding="utf-8") as handle:
            json.dump(run_log, handle, indent=2, ensure_ascii=False)
        print("Run log saved to:", log_path)

    print("\nGame summary")
    print("Questions answered:", question_count)
    print("Correct answers:", correct_count)
    print("Final earnings:", game.earned_amount)
    return game, run_log


final_game, final_run_log = play_actual_simple_rag_game()